In [ ]:
#pip install transformers datasets sacrebleu sentencepiece accelerate

In [59]:
import pandas as pd
import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.layers import TextVectorization
import numpy as np

In [60]:
tf.random.set_seed(42)
np.random.seed(42)

In [61]:
train_df = pd.read_csv("/home/sibel/Langue-wu/Data/Corpus_aligné/train.csv")
dev_df   = pd.read_csv("/home/sibel/Langue-wu/Data/Corpus_aligné/dev.csv")
test_df  = pd.read_csv("/home/sibel/Langue-wu/Data/Corpus_aligné/test.csv")

train_df.head()

,mandarin,wu
0,更不会对美接触过得人有感觉！,更加伐会对没接触过呃拧有感觉！
1,我没找啊！我就说没几个长的好看的,吾没寻啊！吾就讲没记几个长勒好看呃
2,哈哈哈 祝您开心！,哈哈 祝侬开心！
3,我哭了 所以才问你,吾哭了 所以才再问侬
4,给他讲清道理 不厌烦的讲,帮伊讲清道理 伐厌烦呃讲


In [68]:
with open("sp_corpus.txt", "w", encoding="utf-8") as f:
    for t in train_df["wu"]:
        f.write(t + "\n")
    for t in train_df["mandarin"]:
        f.write(t + "\n")

In [69]:
# on fait la vectorization de mandarin (source)
src_vectorizer = TextVectorization(
    max_tokens=4000,
    output_mode="int",
    standardize=None,
    split="character",
)

# la vectorzation de wu (cible)
tgt_vectorizer = TextVectorization(
    max_tokens=4000,
    output_mode="int",
    standardize=None,
    split="character",
)

src_vectorizer.adapt(train_df["mandarin"].astype(str).values)
tgt_vectorizer.adapt(train_df["wu"].astype(str).values)

In [70]:
SRC_VOCAB_SIZE = len(src_vectorizer.get_vocabulary())
TGT_VOCAB_SIZE = len(tgt_vectorizer.get_vocabulary())

print(f"size vocab de mandarin: {SRC_VOCAB_SIZE}")
print(f"size vocab de wu: {TGT_VOCAB_SIZE}")

size vocab de mandarin: 1604
size vocab de wu: 1591


In [71]:
PAD_ID = 0

In [72]:
def prepare_dataset(df, batch_size=32):
    # vectorization
    src_texts = df["mandarin"].astype(str).values
    tgt_texts = df["wu"].astype(str).values
    
    src = src_vectorizer(src_texts)
    tgt = tgt_vectorizer(tgt_texts)
    
    decoder_input = tgt[:, :-1]
    decoder_output = tgt[:, 1:]
    
    dataset = tf.data.Dataset.from_tensor_slices((
        {
            "encoder_input": src,
            "decoder_input": decoder_input
        },
        decoder_output
    ))
    
    dataset = dataset.shuffle(1000).batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return dataset

In [73]:
train_ds = prepare_dataset(train_df, batch_size=32)
dev_ds = prepare_dataset(dev_df, batch_size=32)